# Modelling an interest rate swap and handling instrument events

This Notebook demonstrates the concepts described in the following KBs:

1. Mastering an `InterestRateSwap` instrument, establishing a position, and performing a valuation: https://support.lusid.com/docs/modelling-interest-rate-swaps-in-lusid
2. Handling instrument events: https://support.lusid.com/docs/handling-instrument-events-for-interest-rate-swaps

Note that the simple vanilla swap in this Notebook is extended to exchange notionals at maturity (per a cross-currency swap) in order to demonstrate `SwapPrincipalEvent`.

* Fixed Leg: Receive, 5%
* Floating leg: Pay, SOFR compounded index
* Start date: 15 Jan 2025
* Maturity date: 15 Jan 2030
* Transaction date: 15 Jan 2025

## Setup

In [1]:
import os
import pandas as pd
import json
import uuid
from IPython.core.display import HTML
import logging
from datetime import datetime, timezone, timedelta
logging.basicConfig(level = logging.INFO)

import finbourne.sdk.services.lusid.api as la
import finbourne.sdk.services.lusid.models as lm

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.lpt.lpt import to_date

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)
    
# Confirm success by printing SDK version
api_status = pd.DataFrame(api_factory.build(la.ApplicationMetadataApi).get_lusid_versions().to_dict())
display(api_status)

,apiVersion,buildVersion,excelVersion,links
0,v0,0.6.16065.0,0.5.3666,"{'relation': 'RequestLogs', 'href': 'https://j..."


In [2]:
# Build all the required APIs
try:
    instruments_api = api_factory.build(la.InstrumentsApi)
    instrument_events_api = api_factory.build(la.InstrumentEventsApi)
    instrument_event_type_api = api_factory.build(la.InstrumentEventTypesApi)
    corporate_action_sources_api = api_factory.build(la.CorporateActionSourcesApi)
    aggregation_api = api_factory.build(la.AggregationApi)
    recipe_api = api_factory.build(la.ConfigurationRecipeApi)
    quotes_api = api_factory.build(la.QuotesApi)
    property_definition_api = api_factory.build(la.PropertyDefinitionsApi)
    transaction_portfolios_api = api_factory.build(la.TransactionPortfoliosApi)
    portfolios_api = api_factory.build(la.PortfoliosApi)
    transaction_config_api = api_factory.build(la.TransactionConfigurationApi)
    print("All APIs built correctly")
except ApiException as e:
    print(e)

All APIs built correctly


## Create a scope and code for entities in the Notebook

Keep data segregated from other data in LUSID.

In [3]:
module_scope = "FBNTutorials"
module_code = "VanillaIRS"
print(f"'{module_scope}\\{module_code}' scope and code created.")

'FBNTutorials\VanillaIRS' scope and code created.


## Create property types

Create SHKs to split out cashflow and principal payments from other cash balances in holding reports.

In [4]:
def create_property_type(property_domain, property_scope, property_code, data_type):
    property_type_request = lm.CreatePropertyDefinitionRequest(
        domain = property_domain,
        scope = property_scope,
        code = property_code,
        display_name = property_code,
        data_type_id = lm.ResourceId(scope = "system", code = data_type)
    )

    try:
        property_type_response = property_definition_api.create_property_definition(
            create_property_definition_request = property_type_request
        )
        print(f"Property type created with the following key: {property_type_response.key}")
        return property_type_response.key
    except ApiException as e:
        if json.loads(e.body)["name"] == "PropertyAlreadyExists":
            logging.info(
                f"Property type with the following key already exists: {property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"
            )  
        return f"{property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"

In [5]:
cashflow_shk = create_property_type("Transaction", "SHKs", "SwapCashflow", "string")

INFO:root:Property type with the following key already exists: Transaction/SHKs/SwapCashflow


In [6]:
principal_shk = create_property_type("Transaction", "SHKs", "SwapPrincipal", "string")

INFO:root:Property type with the following key already exists: Transaction/SHKs/SwapPrincipal


## Create transaction types and sides

Required for both purchase transactions and for transactions automatically generated by instrument events.

All created in a custom transaction type scope, which must be registered with the portfolio in which transactions are loaded.

In [7]:
def check_TT(tt, scope):
    try:
        tt_response = transaction_config_api.get_transaction_type(source = f"default", type = tt, scope=scope)
        print(f"\n{tt} transaction type:")
        display(lusid_response_to_data_frame(tt_response.aliases))
        display(lusid_response_to_data_frame(tt_response.movements))
        display(lusid_response_to_data_frame(tt_response.calculations))
    except ApiException as e:
        print(e)
        
def check_side(side, scope):
    try:
        side_response = transaction_config_api.get_side_definition(scope = scope, side = side)
        print(f"\n{side} side:")
        side_response_df = lusid_response_to_data_frame(side_response).transpose()
        side_response_df.drop(side_response_df.filter(regex='links').columns, axis=1, inplace=True)
        display(side_response_df)  
    except ApiException as e:
        print(e)

### Create sides

Must be created before transaction types. Recreate the built-in `Side1` and `Side2` in the custom transaction type scope.

In [8]:
# Recreate Side1 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TradeAmount"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [9]:
# Recreate Side2 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:SettleCcy",
    currency = "Txn:SettlementCurrency",
    rate = "SettledToPortfolioRate",
    units = "Txn:TotalConsideration",
    amount = "Txn:TotalConsideration"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side2",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create an `EnterIRS` transaction type (to establish positions)

In [10]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "EnterIRS",
            description = "Open the contract",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            side = "Side1",
            direction = 1
        )
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "EnterIRS",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create a `SwapCashFlow` transaction type (to handle `SwapCashFlowEvent`)

In [11]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "SwapCashFlow",
            description = "Transaction type for transactions automatically generated by SwapCashFlowEvent",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Pay/receive cashflow",
            movement_types = "CashCommitment",
            side = "Side2",
            direction = 1,
            mappings = [
                lm.TransactionTypePropertyMapping (
                    property_key = f"{cashflow_shk}",
                    set_to = "CashFlow",
                )
            ],
        ),
        lm.TransactionTypeMovement(
            name = "Report as a flow of value in/out",
            movement_types = "Carry",
            side = "Side2",
            direction = 1
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "SwapCashFlow",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create a `SwapPrincipal` transaction type (to handle `SwapPrincipalEvent`)

Note `SwapPrincipalEvent` is only emitted for a leg if `notionalExchangeType` is not `None` in the instrument definition.

In [12]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "SwapPrincipal",
            description = "Transaction type for transactions automatically generated by SwapPrincipalEvent",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Pay/receive cashflow",
            movement_types = "CashCommitment",
            side = "Side2",
            direction = 1,
            mappings = [
                lm.TransactionTypePropertyMapping (
                    property_key = f"{principal_shk}",
                    set_to = "Principal",
                )
            ],
        ),
        lm.TransactionTypeMovement(
            name = "Report as a flow of value in/out",
            movement_types = "Carry",
            side = "Side2",
            direction = 1
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "SwapPrincipal",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create a `Maturity` transaction type (to handle `MaturityEvent`)

In [13]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "Maturity",
            description = "Type for maturity event for various instruments",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Set holding to zero",
            movement_types = "StockMovement",
            direction = -1,
            side = "Side1",
        ),
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "Maturity",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [14]:
check_side("Side1", f"{module_scope}{module_code}")
check_side("Side2", f"{module_scope}{module_code}")

check_TT("EnterIRS", f"{module_scope}{module_code}")
check_TT("SwapCashFlow", f"{module_scope}{module_code}")
check_TT("SwapPrincipal", f"{module_scope}{module_code}")
check_TT("Maturity", f"{module_scope}{module_code}")


Side1 side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Side1,Txn:LusidInstrumentId,Txn:TradeCurrency,Txn:TradeToPortfolioRate,Txn:Units,Txn:TradeAmount,0,None



Side2 side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Side2,Txn:SettleCcy,Txn:SettlementCurrency,SettledToPortfolioRate,Txn:TotalConsideration,Txn:TotalConsideration,0,None



EnterIRS transaction type:


...

""



Maturity transaction type:


,type,description,transaction_class,transaction_roles,is_default
0,Maturity,Type for maturity event for various instruments,Basic,AllRoles,False


,movement_types,side,direction,properties,mappings,name,movement_options,condition,settlement_mode
0,StockMovement,Side1,-1,{},[],Set holding to zero,[],,Internal


""


## Master an instrument

See https://support.lusid.com/docs/modelling-interest-rate-swaps-in-lusid#mastering-an-instrument.

In [15]:
def master_instrument(id):

    start = to_date("2025-01-15")
    maturity = to_date("2030-01-15")
    notional = 100000
    
    instrument_request = {
        id: lm.InstrumentDefinition(
            name = id,
            identifiers = {"ClientInternal": lm.InstrumentIdValue(value = id)},
            definition = lm.InterestRateSwap(
                instrumentType="InterestRateSwap",
                startDate=start,
                maturityDate = maturity,
                legs = [
                    lm.FixedLeg(
                        instrumentType="FixedLeg",
                        startDate=start,
                        maturityDate=maturity,
                        notional = notional,
                        legDefinition=lm.LegDefinition(
                            notionalExchangeType="Final", # If None, SwapPrincipalEvent is not emitted
                            payReceive="Receive",
                            rateOrSpread=0.05,   # This is a rate for a fixed leg
                            stubType="None",
                            conventions=lm.FlowConventions(
                                currency = "USD",
                                payment_frequency = "12M",
                                day_count_convention = "Actual365",
                                roll_convention = "15", # When the period starts/ends.
                                payment_calendars = [],
                                reset_calendars = []
                            )
                        )
                    ),
                    lm.FloatingLeg(
                        instrumentType="FloatingLeg",
                        startDate=start,
                        maturityDate=maturity,
                        notional = notional,
                        legDefinition=lm.LegDefinition(
                            notionalExchangeType="Final", # If None, SwapPrincipalEvent is not emitted
                            payReceive="Pay",
                            rateOrSpread=0, # This is a spread for a floating leg
                            stubType="None",
                            conventions=lm.FlowConventions(
                                currency = "USD",
                                payment_frequency = "12M",
                                day_count_convention = "Actual365",
                                roll_convention = "15",
                                payment_calendars = [],
                                reset_calendars = []
                            ),
                            index_convention = lm.IndexConvention(
                                fixing_reference = "USD-SOFR-COMPOUNDED-INDEX",
                                publication_day_lag = 0,
                                payment_tenor = '1D', 
                                day_count_convention= 'Actual365',
                                currency = 'USD', 
                                index_name = 'USD-SOFR-COMPOUNDED-INDEX'
                            ),
                            resetConvention="InArrears",
                            compounding=lm.Compounding(
                                compoundingMethod="CompoundedIndex",
                                spreadCompoundingMethod="SpreadExclusive",
                                resetFrequency="1D",
                                #calculationShiftMethod="ObservationPeriodShift"
                            )
                        )
                    )
                ]
            )
        )
    }
    
    try:
        instrument_response = instruments_api.upsert_instruments(
            request_body = instrument_request,
            scope = f"{module_scope}{module_code}"
        )
        # Return LUID from (only) instrument object
        return list(instrument_response.values.values())[0].lusid_instrument_id 
    except ApiException as e:
        print(e)

In [16]:
luid_dict = {}
luid_dict["IRS-15Jan30"] = master_instrument("IRS-15Jan30")

for k, v in luid_dict.items():
    print(f"{k}: {v}")

IRS-15Jan30: LUID_00003H2S


In [17]:
def list_instrs():
    instr_response = instruments_api.list_instruments(scope=f"{module_scope}{module_code}")
    instr_response_df = lusid_response_to_data_frame(instr_response, use_camel_case=True)
    instr_response_df.drop(instr_response_df.filter(regex='version|href|staged').columns, axis=1, inplace=True)
    display(instr_response_df.transpose())

list_instrs()

,0
scope,FBNTutorialsVanillaIRS
lusidInstrumentId,LUID_00003H2S
name,IRS-15Jan30
identifiers.ClientInternal,IRS-15Jan30
identifiers.LusidInstrumentId,LUID_00003H2S
properties,[]
instrumentDefinition.instrumentType,InterestRateSwap
instrumentDefinition.startDate,2025-01-15T00:00:00Z
instrumentDefinition.maturityDate,2030-01-15T00:00:00Z
instrumentDefinition.isNonDeliverable,False


## Create recipes

Multiple valid pricing models are valid for an IRS: https://support.lusid.com/docs/modelling-interest-rate-swaps-in-lusid#valuing-your-position

One must be specified as a portfolio recipe to enable instrument events. Any can be used as a valuation recipe.

In [18]:
def extractPricingModelInfo(luid):
    models = instruments_api.get_existing_instrument_models(
        identifier = luid,
        instrument_scope = f"{module_scope}{module_code}"
    )
    models_df = lusid_response_to_data_frame(models).transpose()
    models_df.drop(models_df.filter(regex='links').columns, axis=1, inplace=True)
    display(models_df)

extractPricingModelInfo(luid_dict["IRS-15Jan30"])

,instrument_id,supported_models.0,supported_models.1,supported_models.2,supported_models.3,supported_models.4
response_values,LUID_00003H2S,SimpleStatic,Discounting,VendorDefault,ConstantTimeValueOfMoney,OverrideOnlyPricer


In [19]:
# Universal market rules
myMarketRules = [
    # Look up FX spot rates in the LUSID quote store, if needed
    lm.MarketDataKeyRule(
        key = "Fx.CurrencyPair.*",
        supplier = "Lusid",
        data_scope = f"{module_scope}{module_code}",
        quote_type = "Rate",
        field = "mid",
    ),
    # Look up clean market prices
    lm.MarketDataKeyRule(
        key = "Quote.*.*",
        supplier = "Lusid",
        data_scope = f"{module_scope}{module_code}",
        quote_type = "Price",
        field = "mid",
    ),
    # Look up dirty market prices
    lm.MarketDataKeyRule(
        key = "Quote.*.*",
        supplier = "Lusid",
        data_scope = f"{module_scope}{module_code}",
        quote_type = "DirtyPrice",
        field = "mid",
    ),
    # Look up interest rate fixings for floating leg
    lm.MarketDataKeyRule(
        key = "Quote.RIC.*",
        supplier = "Lusid",
        data_scope = f"{module_scope}{module_code}",
        quote_type = "Index",
        field = "mid",
    )
]

### Create a SimpleStatic recipe

In [20]:
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as the portfolio
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-SimpleStatic",
    description = "A recipe to value an IRS using SimpleStatic",
    market = lm.MarketContext(
        market_rules = myMarketRules
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create a CTVoM recipe with a single leg (combined)

In [21]:
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as the portfolio
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-CTVOMOneLeg",
    description = "A recipe to value an IRS using CTVOM with the legs combined",
    market = lm.MarketContext(
        market_rules = myMarketRules
    ),
    pricing=lm.PricingContext(
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="ConstantTimeValueOfMoney",
                instrument_type="InterestRateSwap"
            )
        ],
        options=lm.PricingOptions(
            produceSeparateResultForLinearOtcLegs=False
        )
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create a CTVoM recipe with separate legs

In [22]:
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as the portfolio
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-CTVOMTwoLegs",
    description = "A recipe to value an IRS using CTVOM with separate legs",
    market = lm.MarketContext(
        market_rules = myMarketRules
    ),
    pricing=lm.PricingContext(
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="ConstantTimeValueOfMoney",
                instrument_type="InterestRateSwap"
            )
        ],
        options=lm.PricingOptions(
            produceSeparateResultForLinearOtcLegs=True
        )
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [23]:
# Confirm recipe upsert and show the many options that are automatically set to default values by LUSID.
config_recipe = recipe_api.list_configuration_recipes(filter=f"value.scope eq '{module_scope}' and value.code startswith '{module_code}'")
config_recipe_df = lusid_response_to_data_frame(config_recipe)
display(config_recipe_df.transpose())

,0,1,2
value.scope,FBNTutorials,FBNTutorials,FBNTutorials
value.code,VanillaIRS-CTVOMOneLeg,VanillaIRS-SimpleStatic,VanillaIRS-CTVOMTwoLegs
value.market.market_rules.0.key,Fx.CurrencyPair.*,Fx.CurrencyPair.*,Fx.CurrencyPair.*
value.market.market_rules.0.supplier,Lusid,Lusid,Lusid
value.market.market_rules.0.data_scope,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS
value.market.market_rules.0.quote_type,Rate,Rate,Rate
value.market.market_rules.0.var_field,mid,mid,mid
value.market.market_rules.0.price_source,,,
value.market.market_rules.0.source_system,Lusid,Lusid,Lusid
value.market.market_rules.0.fall_through_on_access_denied,False,False,False


## Set up a USD transaction portfolio

The portfolio recipe is set to one of the recipes created above to enable instrument events, and the transaction type scope is registered.

In [24]:
def create_portfolio(name):
    portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name = f"{name} portfolio",
        code = f"{module_code}-{name}",
        # Set the portfolio currency
        base_currency = "USD",
        # Must be before first transaction recorded
        created = datetime.strptime("2024-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        # Attempt to resolve transactions to instruments in the custom scope before falling back to the default scope
        instrument_scopes = [f"{module_scope}{module_code}"],
        # Register transaction type scope
        transactionTypeScope=f"{module_scope}{module_code}",
        # Register portfolio recipe        
        instrumentEventConfiguration=lm.InstrumentEventConfiguration(
            recipeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-SimpleStatic"
            )
        )
    )

    try:
        portfolio_response=transaction_portfolios_api.create_portfolio(
            scope = module_scope,
            create_transaction_portfolio_request = portfolio_request
        )
        print(f"Portfolio with display name '{portfolio_response.display_name}' created effective {str(portfolio_response.created)}")
    except ApiException as e:
        print(e)

In [25]:
ports = ["VanillaIRS"]
# ports = ["VanillaIRS", "XCCYIRS"]

In [26]:
for port in ports:
    create_portfolio(port)

Portfolio with display name 'VanillaIRS portfolio' created effective 2024-01-01 00:00:00+00:00


### Confirm portfolio details

In [27]:
def get_port_details(port):
    portfolio_response = transaction_portfolios_api.get_details(scope = module_scope, code = f"{module_code}-{port}")
    portfolio_response_df = lusid_response_to_data_frame(portfolio_response).transpose()
    # Drop some noisy columns
    portfolio_response_df.drop(portfolio_response_df.filter(regex='version|href|staged|links|settlement').columns, axis=1, inplace=True)
    display(portfolio_response_df.transpose())
    
for port in ports:
    get_port_details(port)

,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,VanillaIRS-VanillaIRS
base_currency,USD
corporate_action_source_id,None
sub_holding_keys,[]
instrument_scopes.0,FBNTutorialsVanillaIRS
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsVanillaIRS
cash_gain_loss_calculation_date,Default


## Load transactions to establish positions

See https://support.lusid.com/docs/modelling-interest-rate-swaps-in-lusid#booking-a-transaction-to-establish-a-position.

In [28]:
# Create convenience function
def create_transactions(port, txnid, tttype, luid, tradedate, quantity, price, ccy):
    
    create_txn_request = {
        "number_one": lm.TransactionRequest(
            transaction_id=txnid,
            type=tttype,
            instrument_identifiers = {"Instrument/default/LusidInstrumentId": luid},
            transaction_date=tradedate,
            settlement_date=tradedate,
            units=quantity,
            transaction_currency = ccy,
            transaction_price = lm.TransactionPrice(
                price = price,
                type = "Price"
            ),
            total_consideration = lm.CurrencyAndAmount(
                amount = quantity * price,
                currency = ccy,
            )
        )
    }
    
    try:
        create_txn_response = transaction_portfolios_api.batch_upsert_transactions(
            scope = f"{module_scope}",
            code = f"{module_code}-{port}",
            success_mode="Partial",
            request_body = create_txn_request
        )
        print(create_txn_response.failed) if create_txn_response.failed else print("Success")
    except ApiException as e:
        print(e)

In [29]:
create_transactions("VanillaIRS", "Txn01", "EnterIRS", luid_dict["IRS-15Jan30"], "2025-01-15", 1, 0, "USD")

Success


### Confirm positions and audit output transactions

In [30]:
def get_portfolio_holdings(port, date):      
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))
    
    try:
        get_holdings_response = transaction_portfolios_api.get_holdings(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            property_keys=["Instrument/default/Name"]
        )
        get_holdings_response_df = lusid_response_to_data_frame(get_holdings_response)
        get_holdings_response_df.rename(columns = {
            "properties.Instrument/default/Name.value.label_value": "instrument"}, inplace = True)        
        # Drop some noisy columns
        get_holdings_response_df.drop(get_holdings_response_df.filter(regex='properties|sub_holding_keys').columns, axis=1, inplace=True)
        display(get_holdings_response_df)
    except ApiException as e:
        print(e)

In [31]:
def get_output_transactions(port, start, end, all_data):
    try:
        output_transactions_response = transaction_portfolios_api.build_transactions(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            transaction_query_parameters = lm.TransactionQueryParameters(
                start_date = datetime.strptime(start, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                end_date = datetime.strptime(end, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat()
            )
        )
        output_transactions_response_df = lusid_response_to_data_frame(output_transactions_response, use_camel_case=True)
        if all_data == False:
            output_transactions_response_df = output_transactions_response_df[["type", "sourceType", "settlementDate", "transactionId", "totalConsideration.amount"]]
            display(output_transactions_response_df)
        else:
            display(output_transactions_response_df.transpose())
    except ApiException as e:
        print(e)

In [32]:
for port in ports:
    print(f"\n{port}")
    get_portfolio_holdings(port, "2025-01-15 00:00:00")   # Instrument start date (also trade/settlement date)


VanillaIRS


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsVanillaIRS,LUID_00003H2S,IRS-15Jan30,P,1.00,1.00,0.00,USD,0.00,USD,USD,Position,81333661,0.00,USD,0.00,USD,0.00,USD,0.00,USD,0.00,USD,[],0.00,0.00


In [33]:
for port in ports:
    print(f"\n{port}")
    get_output_transactions(port, "2024-01-01", "2025-01-15", True)


VanillaIRS


,0
transactionId,Txn01
type,EnterIRS
description,Open the contract
instrumentIdentifiers.Instrument/default/LusidInstrumentId,LUID_00003H2S
instrumentScope,FBNTutorialsVanillaIRS
instrumentUid,LUID_00003H2S
transactionDate,2025-01-15T00:00:00Z
settlementDate,2025-01-15T00:00:00Z
units,1.00
transactionAmount,0.00


## Instrument events

LUSID emits `SwapCashflowEvent` for each leg on each payment date. 

For a fixed leg, no market data is required, and a corresponding output transaction is automatically generated. 

For a floating leg, a corresponding output transaction is only generated if the correct number of interest rate fixings have been loaded in as market data; see https://support.lusid.com/docs/modelling-interest-rate-swaps-in-lusid#providing-interest-rate-fixings.

`SwapPrincipalEvent` is only emitted for a leg if `notionalExchangeType` is not `None` in the instrument definition.

### Examine state before market data loaded

Since no market data is loaded, LUSID does emit `SwapCashflowEvent` for the floating `Pay` leg but each instance has a market data failure error that means no corresponding output transaction is generated.

In [34]:
def query_instr_events(port):
    
    query_id_request = lm.QueryApplicableInstrumentEventsRequest(
        window_start = datetime.strptime("2015-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        window_end = datetime.strptime("2031-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        effective_at = datetime.strptime("2031-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        portfolio_entity_ids = [
            lm.PortfolioEntityId(
                scope = module_scope,
                code = f"{module_code}-{port}",
            )
        ],
        forecasting_recipe_id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-SimpleStatic"
        )
    )
    
    try:
        query_id_response = instrument_events_api.query_applicable_instrument_events(
            query_applicable_instrument_events_request = query_id_request,
            limit = 200
        )
        display(lusid_response_to_data_frame(query_id_response, use_camel_case=True).transpose())
    except ApiException as e:
        print(e)

In [35]:
pd.set_option('display.max_colwidth', 200)  
for port in ports:
    print(f"\n{port}")
    query_instr_events(port)


VanillaIRS


,0,1,2,3,4,5,6,7,8,9,10,11,12
portfolioId.scope,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials
portfolioId.code,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS
holdingId,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661
lusidInstrumentId,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S
instrumentScope,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS
instrumentType,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap
instrumentEventType,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapPrincipalEvent,SwapPrincipalEvent,MaturityEvent
instrumentEventId,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_MaturityEvent_20300115
generatedEvent.instrumentEventId,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_MaturityEvent_20300115
generatedEvent.instrumentIdentifiers.Instrument/default/ClientInternal,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30


By default, output transactions of type `SwapCashFlow` are only generated for the fixed `Receive` leg.

In [36]:
pd.set_option('display.max_colwidth', 200)  
for port in ports:
    print(f"Summary")
    get_output_transactions(port, "2024-01-01", "2030-12-31", False)

Summary


,type,sourceType,settlementDate,transactionId,totalConsideration.amount
0,EnterIRS,InputTransaction,2025-01-15T00:00:00Z,Txn01,0.00
1,SwapCashFlow,InstrumentEvent,2026-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20260115-81333661,"5,000.00"
2,SwapCashFlow,InstrumentEvent,2027-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20270115-81333661,"5,000.00"
3,SwapCashFlow,InstrumentEvent,2028-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20280115-81333661,"5,000.00"
4,SwapCashFlow,InstrumentEvent,2029-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20290115-81333661,"5,013.70"
5,SwapCashFlow,InstrumentEvent,2030-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20300115-81333661,"5,000.00"
6,SwapPrincipal,InstrumentEvent,2030-01-15T00:00:00Z,LUID_00003H2S_SwapPrincipalEvent_Leg-1_Receive_USD_20300115-81333661,"100,000.00"
7,SwapPrincipal,InstrumentEvent,2030-01-15T00:00:00Z,LUID_00003H2S_SwapPrincipalEvent_Leg-2_Pay_USD_20300115-81333661,"-100,000.00"
8,Maturity,InstrumentEvent,2030-01-15T00:00:00Z,LUID_00003H2S_MaturityEvent_20300115-81333661,0.00


### Load interest rate fixings for the floating leg

In [37]:
def load_quotes(id_type, id, q_type, date, price, ccy, scale):
    if date == "today":
        date = str(datetime.datetime.now().replace(microsecond=0))

    quotes = {
        # Each quote must be upserted with an ephemeral key (uuid in this case), to track errors in the response
        str(uuid.uuid4()): lm.UpsertQuoteRequest(
            quote_id = lm.QuoteId(
                quote_series_id = lm.QuoteSeriesId(
                    # Must be one of the valid financial data vendor 'provider' values
                    provider = "Lusid",
                    instrument_id_type = id_type,
                    instrument_id = id,
                    quote_type = q_type,
                    # Case sensitive: the field value must match that of the equivalent recipe field exactly
                    field = "mid",
                ),
                effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            ),
            metric_value = lm.MetricValue(value = price, unit = ccy),
            scale_factor = scale,
        )
    }

    try:
        upsert_quotes_response = quotes_api.upsert_quotes(scope = f"{module_scope}{module_code}", request_body = quotes)    
        if upsert_quotes_response.failed == {}:
            print(f"{id} price for {date} successfully loaded into LUSID.")
        else:
            print(f"Some failures occurred. {len(upsert_quotes_response.failed)} prices did not get loaded into LUSID.")
    except ApiException as e:
        print(e)

In [38]:
# Fixing data for payment dates that have occurred at the time of writing are taken from https://www.newyorkfed.org/markets/reference-rates/sofr-averages-and-index. Subsequent fixing data is fabricated.
# If the interest rate period starts on a Saturday or Sunday, LUSID requires a fixing for the previous business day, so in this case on Fri 14 Jan 2028 (since 15 Jan 2028 is a Saturday).
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2025-01-15 00:00:00", 1.17692687, "USD", 100) # Instrument start date/transaction date
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2026-01-15 00:00:00", 1.22834029, "USD", 100) # First payment date
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2027-01-15 00:00:00", 1.24232876, "USD", 100) # Second payment date
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2028-01-14 00:00:00", 1.26928272, "USD", 100) # Day before third payment date.
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2029-01-15 00:00:00", 1.28928272, "USD", 100) # Fourth payment date
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2030-01-15 00:00:00", 1.30928272, "USD", 100) # Fifth payment date/instrument maturity date

USD-SOFR-COMPOUNDED-INDEX price for 2025-01-15 00:00:00 successfully loaded into LUSID.
USD-SOFR-COMPOUNDED-INDEX price for 2026-01-15 00:00:00 successfully loaded into LUSID.
USD-SOFR-COMPOUNDED-INDEX price for 2027-01-15 00:00:00 successfully loaded into LUSID.
USD-SOFR-COMPOUNDED-INDEX price for 2028-01-14 00:00:00 successfully loaded into LUSID.
USD-SOFR-COMPOUNDED-INDEX price for 2029-01-15 00:00:00 successfully loaded into LUSID.
USD-SOFR-COMPOUNDED-INDEX price for 2030-01-15 00:00:00 successfully loaded into LUSID.


In [39]:
def list_quotes():
    try:
        quotes_response = quotes_api.list_quotes_for_scope(f"{module_scope}{module_code}")
        quotes_response_df = lusid_response_to_data_frame(quotes_response)
        display(quotes_response_df)        
    except ApiException as e:
        print(e)

list_quotes()

,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2030-01-15T00:00:00.0000000+00:00,1.31,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:52.430545+00:00,100.00
1,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2029-01-15T00:00:00.0000000+00:00,1.29,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:52.195056+00:00,100.00
2,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2028-01-14T00:00:00.0000000+00:00,1.27,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:51.971124+00:00,100.00
3,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2027-01-15T00:00:00.0000000+00:00,1.24,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:51.751909+00:00,100.00
4,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2026-01-15T00:00:00.0000000+00:00,1.23,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:51.062031+00:00,100.00
5,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2025-01-15T00:00:00.0000000+00:00,1.18,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:50.834577+00:00,100.00


### Examine state after market data loaded

Now there is no market data failure in `SwapCashflowEvent` for the floating `Pay` leg.

In [40]:
for port in ports:
    print(f"\n{port}")
    query_instr_events(port)


VanillaIRS


,0,1,2,3,4,5,6,7,8,9,10,11,12
portfolioId.scope,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials
portfolioId.code,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS,VanillaIRS-VanillaIRS
holdingId,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661,81333661
lusidInstrumentId,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S,LUID_00003H2S
instrumentScope,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS,FBNTutorialsVanillaIRS
instrumentType,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap,InterestRateSwap
instrumentEventType,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapCashFlowEvent,SwapPrincipalEvent,SwapPrincipalEvent,MaturityEvent
instrumentEventId,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_MaturityEvent_20300115
generatedEvent.instrumentEventId,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20260115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20270115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20280115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20290115,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-1_Receive_USD_20300115,LUID_00003H2S_SwapPrincipalEvent_Leg-2_Pay_USD_20300115,LUID_00003H2S_MaturityEvent_20300115
generatedEvent.instrumentIdentifiers.Instrument/default/ClientInternal,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30,IRS-15Jan30


And now output transactions of type `SwapCashFlow` are also generated for the floating `Pay` leg.

In [41]:
for port in ports:
    print(f"Summary")
    get_output_transactions(port, "2024-01-01", "2030-12-31", False)

Summary


,type,sourceType,settlementDate,transactionId,totalConsideration.amount
0,EnterIRS,InputTransaction,2025-01-15T00:00:00Z,Txn01,0.00
1,SwapCashFlow,InstrumentEvent,2026-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20260115-81333661,"5,000.00"
2,SwapCashFlow,InstrumentEvent,2026-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20260115-81333661,"-4,368.45"
3,SwapCashFlow,InstrumentEvent,2027-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20270115-81333661,"5,000.00"
4,SwapCashFlow,InstrumentEvent,2027-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20270115-81333661,"-1,138.81"
5,SwapCashFlow,InstrumentEvent,2028-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20280115-81333661,"5,000.00"
6,SwapCashFlow,InstrumentEvent,2028-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20280115-81333661,"-2,169.63"
7,SwapCashFlow,InstrumentEvent,2029-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20290115-81333661,"5,013.70"
8,SwapCashFlow,InstrumentEvent,2029-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-2_Pay_USD_20290115-81333661,"-1,575.69"
9,SwapCashFlow,InstrumentEvent,2030-01-15T00:00:00Z,LUID_00003H2S_SwapCashFlowEvent_Leg-1_Receive_USD_20300115-81333661,"5,000.00"


### Examine holdings over the lifetime of the contract

In [42]:
for port in ports:
    print(f"\n{port}")
    get_portfolio_holdings(port, "2025-01-15 00:00:00")   # Instrument start date (also trade/settlement date)
    get_portfolio_holdings(port, "2026-01-15 00:00:00")   # 1st payment date
    get_portfolio_holdings(port, "2027-01-15 00:00:00")   # 2nd payment date
    get_portfolio_holdings(port, "2028-01-15 00:00:00")   # 3rd payment date
    get_portfolio_holdings(port, "2029-01-15 00:00:00")   # 4th payment date
    get_portfolio_holdings(port, "2030-01-15 00:00:00")   # Instrument maturity date


VanillaIRS


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsVanillaIRS,LUID_00003H2S,IRS-15Jan30,P,1.00,1.00,0.00,USD,0.00,USD,USD,Position,81333661,0.00,USD,0.00,USD,0.00,USD,0.00,USD,0.00,USD,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsVanillaIRS,LUID_00003H2S,IRS-15Jan30,P,1.00,1.00,0.00,USD,0.00,USD,USD,Position,81333661,0.00,USD,0.00,USD,0.00,USD,0.00,USD,0.00,USD,[],0.00,0.00
1,default,CCY_USD,USD,B,631.55,631.55,631.55,USD,631.55,USD,USD,Balance,81333662,0.00,USD,631.55,USD,631.55,USD,0.00,USD,0.00,USD,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsVanillaIRS,LUID_00003H2S,IRS-15Jan30,P,1.00,1.00,0.00,USD,0.00,USD,USD,Position,81333661,0.00,USD,0.00,USD,0.00,USD,0.00,USD,0.00,USD,[],0.00,0.00
1,default,CCY_USD,USD,B,"4,492.74","4,492.74","4,492.74",USD,"4,492.74",USD,USD,Balance,81333662,0.00,USD,"4,492.74",USD,"4,492.74",USD,0.00,USD,0.00,USD,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsVanillaIRS,LUID_00003H2S,IRS-15Jan30,P,1.00,1.00,0.00,USD,0.00,USD,USD,Position,81333661,0.00,USD,0.00,USD,0.00,USD,0.00,USD,0.00,USD,[],0.00,0.00
1,default,CCY_USD,USD,B,"7,323.11","7,323.11","7,323.11",USD,"7,323.11",USD,USD,Balance,81333662,0.00,USD,"7,323.11",USD,"7,323.11",USD,0.00,USD,0.00,USD,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsVanillaIRS,LUID_00003H2S,IRS-15Jan30,P,1.00,1.00,0.00,USD,0.00,USD,USD,Position,81333661,0.00,USD,0.00,USD,0.00,USD,0.00,USD,0.00,USD,[],0.00,0.00
1,default,CCY_USD,USD,B,"10,761.12","10,761.12","10,761.12",USD,"10,761.12",USD,USD,Balance,81333662,0.00,USD,"10,761.12",USD,"10,761.12",USD,0.00,USD,0.00,USD,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,default,CCY_USD,USD,B,"14,209.87","14,209.87","14,209.87",USD,"14,209.87",USD,USD,Balance,81333662,0.00,USD,"14,209.87",USD,"14,209.87",USD,0.00,USD,0.00,USD,[],0.00,0.00


## Value portfolio

See https://support.lusid.com/docs/modelling-interest-rate-swaps-in-lusid#valuing-your-position

Random valuation date: 29 May 2026

In [43]:
def value_instruments(port, date, pnl_window, recipe):
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))

    valuation_request = lm.ValuationRequest(
        # Choose recipe to use
        recipe_id = lm.ResourceId(scope = module_scope, code = f"{module_code}-{recipe}"),
        # Specify metrics (also known as queryable keys) to report useful information
        metrics = [
            lm.AggregateSpec(key="Instrument/InstrumentCategory", op="Value"),
            lm.AggregateSpec(key="Instrument/default/Name", op="Value"),
            lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
            lm.AggregateSpec(key="Valuation/LegIndex", op="Value"),
            lm.AggregateSpec(key="Valuation/LegIdentifier", op="Value"),
            lm.AggregateSpec(key="Valuation/Model/Name", op="Value"),
            lm.AggregateSpec(key="Valuation/EffectiveAt", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Value"),
            lm.AggregateSpec(key="Quotes/PriceOrFXRate", op="Value"),
            lm.AggregateSpec(key="Holding/Cost/Dom", op="Value"),
            lm.AggregateSpec(key="Valuation/CleanPV", op="Value"),
            lm.AggregateSpec(key="Valuation/PV", op="Value"),
            lm.AggregateSpec(key="Valuation/Accrued", op="Value"),
            # lm.AggregateSpec(key="Valuation/Exposure", op="Value"),          
            # lm.AggregateSpec(key="Valuation/CurrentNotional", op="Value"),
            lm.AggregateSpec(key="ProfitAndLoss/Total", op="Value", options={"Window": f"{pnl_window}"}),      
            lm.AggregateSpec(key="ProfitAndLoss/Total/Market", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="ProfitAndLoss/Realised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Unrealised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Total/Other", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="Aggregation/Errors", op="Value"), 
        ],
        # Identify portfolio to value
        portfolio_entity_ids = [lm.PortfolioEntityId(scope = module_scope, code = f"{module_code}-{port}")],
        valuation_schedule = lm.ValuationSchedule(effective_at = date),

    )

    try:
        val_response = aggregation_api.get_valuation(valuation_request = valuation_request)
        val_response_df = pd.json_normalize(val_response.to_dict()["data"], sep='.')
        # Rename columns
        val_response_df.rename(
            columns = {
                "Instrument/InstrumentCategory": "Category",
                "Valuation/Model/Name": "Pricing model",
                "Instrument/default/LusidInstrumentId": "LUID",
                "Instrument/default/Name": "Name",
                "Valuation/EffectiveAt": "Date",
                "Holding/default/Units": "Units",
                "Quotes/PriceOrFXRate": "Price",
                "Quotes/ScaleFactor": "Quote Scale Factor",
                "Holding/Cost/Dom": "Local Cost",
                "Valuation/CleanPV": "Local Clean PV",
                "Valuation/PV": "Local PV",
                "Valuation/Accrued": "Local Accrued Interest",
                "Valuation/Exposure": "Exposure",
                "Valuation/CurrentNotional": "Notional",            
                f"ProfitAndLoss/Total(Window=\"{pnl_window}\")": "Total P&L",
                f"ProfitAndLoss/Total/Market(Window=\"{pnl_window}\")": "Total/Market P&L",
                f"ProfitAndLoss/Realised/Market(Window=\"{pnl_window}\")": "Realised/Market P&L",
                f"ProfitAndLoss/Unrealised/Market(Window=\"{pnl_window}\")": "Unrealised/Market P&L",
                f"ProfitAndLoss/Total/Other(Window=\"{pnl_window}\")": "Total/Other P&L",
                "Aggregation/Errors": "Errors"
            },
            inplace = True,
        )
        val_response_df["Date"] = pd.to_datetime(val_response_df["Date"]).dt.date
        display(val_response_df)
    except ApiException as e:
        print(e)

### CTVoM pricing model

Providing a floating leg has interest rate fixings, no market price is required.

For a compounded index such as the SOFR index, LUSID requires fixings for the start and end of a valuation 'calculation period'; see https://support.lusid.com/docs/modelling-interest-rate-swaps-in-lusid#providing-interest-rate-fixings. 

For a valuation date of 29 May 2026, these are:

* Start of calculation period: 15 Jan 2026
* End of calculation period: 29 May 2026

Note `ProfitAndLoss` keys with a window such as `YTD` might require extra market data: https://support.lusid.com/docs/specifying-a-pnl-window-and-other-metric-options.

In [44]:
# Some may have already been loaded in the instrument events section above, but load again anyway
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2025-01-15 00:00:00", 1.17692687, "USD", 100) # Required for ProfitAndLoss keys with a window
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2026-01-15 00:00:00", 1.22834029, "USD", 100) # Start of calculation period
load_quotes("RIC", "USD-SOFR-COMPOUNDED-INDEX", "Index", "2026-05-29 00:00:00", 1.24508236, "USD", 100) # End of calculation period

list_quotes()

USD-SOFR-COMPOUNDED-INDEX price for 2025-01-15 00:00:00 successfully loaded into LUSID.
USD-SOFR-COMPOUNDED-INDEX price for 2026-01-15 00:00:00 successfully loaded into LUSID.
USD-SOFR-COMPOUNDED-INDEX price for 2026-05-29 00:00:00 successfully loaded into LUSID.


,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2030-01-15T00:00:00.0000000+00:00,1.31,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:52.430545+00:00,100.00
1,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2029-01-15T00:00:00.0000000+00:00,1.29,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:52.195056+00:00,100.00
2,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2028-01-14T00:00:00.0000000+00:00,1.27,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:51.971124+00:00,100.00
3,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2027-01-15T00:00:00.0000000+00:00,1.24,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:51.751909+00:00,100.00
4,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2026-05-29T00:00:00.0000000+00:00,1.25,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:03.255998+00:00,100.00
5,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2026-01-15T00:00:00.0000000+00:00,1.23,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:02.957916+00:00,100.00
6,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2025-01-15T00:00:00.0000000+00:00,1.18,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:02.790462+00:00,100.00


In [45]:
for port in ports:
    print(f"\n{port}")
    value_instruments(port, "2026-05-29", "YTD", "CTVOMOneLeg") 
    value_instruments(port, "2026-05-29", "YTD", "CTVOMTwoLegs")


VanillaIRS


,Category,Name,LUID,Valuation/LegIndex,Valuation/LegIdentifier,Pricing model,Date,Units,Price,Local Cost,Local Clean PV,Local PV,Local Accrued Interest,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,InterestRateSwap,IRS-15Jan30,LUID_00003H2S,None,None,ConstantTimeValueOfMoney,2026-05-29,1.00,1.25,0.00,0.00,"14,086.93",472.63,-882.75,"3,439.14",0.00,"3,439.14","-4,321.89",[]
1,Cash,USD,CCY_USD,None,None,ConstantTimeValueOfMoney,2026-05-29,631.55,1.00,631.55,631.55,631.55,0.00,631.55,0.00,0.00,0.00,631.55,[]


,Category,Name,LUID,Valuation/LegIndex,Valuation/LegIdentifier,Pricing model,Date,Units,Price,Local Cost,Local Clean PV,Local PV,Local Accrued Interest,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,InterestRateSwap,IRS-15Jan30,LUID_00003H2S,1.00,Receive,ConstantTimeValueOfMoney,2026-05-29,1.00,1.00,0.00,"120,013.70","120,013.70","1,835.62","-5,000.00","-2,041.10",0.00,"-2,041.10","-2,958.90",[]
1,InterestRateSwap,IRS-15Jan30,LUID_00003H2S,2.00,Pay,ConstantTimeValueOfMoney,2026-05-29,1.00,1.00,0.00,"-105,926.77","-105,926.77","-1,362.98","4,117.25","5,480.24",0.00,"5,480.24","-1,362.98",[]
2,Cash,USD,CCY_USD,NaN,None,ConstantTimeValueOfMoney,2026-05-29,631.55,1.00,631.55,631.55,631.55,0.00,631.55,0.00,0.00,0.00,631.55,[]


### SimpleStatic pricing model

In addition to interest rate fixings for a floating leg, a market price is required for the valuation date.

In [46]:
load_quotes("LusidInstrumentId", luid_dict["IRS-15Jan30"], "DirtyPrice", "2025-12-31 00:00:00", 0.1, "USD", 1)  # Required for ProfitAndLoss keys with a window
load_quotes("LusidInstrumentId", luid_dict["IRS-15Jan30"], "DirtyPrice", "2026-05-29 00:00:00", 0.09787, "USD", 1)  # Valuation date
list_quotes()

LUID_00003H2S price for 2025-12-31 00:00:00 successfully loaded into LUSID.
LUID_00003H2S price for 2026-05-29 00:00:00 successfully loaded into LUSID.


,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2030-01-15T00:00:00.0000000+00:00,1.31,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:52.430545+00:00,100.00
1,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2029-01-15T00:00:00.0000000+00:00,1.29,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:52.195056+00:00,100.00
2,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2028-01-14T00:00:00.0000000+00:00,1.27,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:51.971124+00:00,100.00
3,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2027-01-15T00:00:00.0000000+00:00,1.24,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:49:51.751909+00:00,100.00
4,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2026-05-29T00:00:00.0000000+00:00,1.25,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:03.255998+00:00,100.00
5,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2026-01-15T00:00:00.0000000+00:00,1.23,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:02.957916+00:00,100.00
6,Lusid,USD-SOFR-COMPOUNDED-INDEX,RIC,Index,mid,4d001317-e7e1-4969-b898-949e41ad0b24,2025-01-15T00:00:00.0000000+00:00,1.18,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:02.790462+00:00,100.00
7,Lusid,LUID_00003H2S,LusidInstrumentId,DirtyPrice,mid,0a659bfb-57ef-4214-a470-fb79fc5a6200,2026-05-29T00:00:00.0000000+00:00,0.10,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:10.957871+00:00,1.00
8,Lusid,LUID_00003H2S,LusidInstrumentId,DirtyPrice,mid,0a659bfb-57ef-4214-a470-fb79fc5a6200,2025-12-31T00:00:00.0000000+00:00,0.10,USD,,,00u91lo2d7X42sdse2p7,2026-06-25 07:50:10.737008+00:00,1.00


In [47]:
for port in ports:
    print(f"\n{port}")
    value_instruments(port, "2026-05-29", "YTD", "SimpleStatic")


VanillaIRS


,Category,Name,LUID,Valuation/LegIndex,Valuation/LegIdentifier,Pricing model,Date,Units,Price,Local Cost,Local Clean PV,Local PV,Local Accrued Interest,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,InterestRateSwap,IRS-15Jan30,LUID_00003H2S,None,None,SimpleStatic,2026-05-29,1.00,0.10,0.00,"9,314.37","9,787.00",472.63,-213.00,"4,108.89",0.00,"4,108.89","-4,321.89",[]
1,Cash,USD,CCY_USD,None,None,ConstantTimeValueOfMoney,2026-05-29,631.55,1.00,631.55,631.55,631.55,0.00,631.55,0.00,0.00,0.00,631.55,[]
